# The Hertzsprung–Russell Diagram with Gaia

### From a raw ESA star catalogue to a map of stellar evolution

---

The **Hertzsprung–Russell (HR) diagram** is one of the most important plots in stellar
astrophysics. It plots stars by colour (temperature) against true brightness, and when you
do that, stars fall into clear families that reveal how they are born, live, and die.

Gaia measured the positions, distances, motions, and colours of around **1.8 billion
stars**. This notebook takes one small slice of that catalogue and builds an HR diagram from
it, step by step.

**What this notebook covers (every plot is interactive — zoom, pan, hover, drag sliders):**

1. The raw HR diagram, straight from the data
2. A cleaned diagram using only high-precision distances
3. An HR diagram coloured by real stellar temperature, with the main sequence, giants, and white dwarfs labelled
4. A slider playground to change the quality cuts and watch the diagram respond
5. Famous stars placed on the diagram, from the Sun to Betelgeuse
6. An animation of how a Sun-like star evolves and dies across the diagram
7. Theoretical isochrones (lines of constant age) for reading a cluster's age
8. A guided project: rebuild everything with data from a different telescope

**How to run:** click the first code cell and press **Shift+Enter** to run it and move to the
next. Repeat down the notebook. That's it.

## 0. Setup — install and import the tools

Run this cell once. On Google Colab everything except `plotly` is already installed; the line
below quietly adds anything missing. Wait for the confirmation message before continuing.

In [ ]:
# If you are on Google Colab or a fresh machine, this installs what's needed.
# It is safe to re-run — it does nothing if the packages already exist.
import sys, subprocess

def _ensure(pkgs):
    import importlib
    missing = []
    for pip_name, import_name in pkgs:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(pip_name)
    if missing:
        print('Installing:', ', '.join(missing), '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=False)

_ensure([
    ('pandas','pandas'), ('numpy','numpy'),
    ('plotly','plotly'), ('ipywidgets','ipywidgets'),
    ('matplotlib','matplotlib'),
])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

pd.set_option('display.max_columns', 500)
# Make Plotly render in both Jupyter and Colab
try:
    import plotly.io as pio
    pio.renderers.default = 'colab' if 'google.colab' in sys.modules else 'notebook'
except Exception:
    pass

print('✅ All tools ready. Versions:')
print('   pandas ', pd.__version__)
print('   numpy  ', np.__version__)
import plotly; print('   plotly ', plotly.__version__)


## Warm-up — the Harvard colour system

Before we touch any real telescope data, here is the single idea the whole HR diagram rests
on: **a star's colour tells us its temperature.**

Astronomers sort stars into seven colour classes — the **Harvard spectral sequence**:

> **O B A F G K M**  —  hottest and bluest → coolest and reddest
>
> *(a common mnemonic: "Oh Be A Fine Girl/Guy, Kiss Me")*

The next three cells are quick demos that make up their own numbers, so they run instantly
and need no data download.

### Demo 1 — what each spectral class actually looks like

This draws the seven classes as coloured swatches, from hot blue stars to cool red ones,
using an approximate blackbody colour for each surface temperature.


In [ ]:
import numpy as np
import plotly.graph_objects as go

# The seven Harvard classes with a representative surface temperature (Kelvin)
classes = ['O','B','A','F','G','K','M']
temps   = [40000, 20000, 8500, 6500, 5500, 4000, 3000]
examples= ['Mintaka','Rigel','Vega','Procyon','the Sun','Arcturus','Betelgeuse']

def blackbody_rgb(T):
    """Very rough Kelvin -> RGB just for illustration (not physically exact)."""
    T = T/100.0
    if T <= 66:
        r = 255
        g = 99.47*np.log(T) - 161.12 if T > 0 else 0
    else:
        r = 329.7*((T-60)**-0.1332)
        g = 288.12*((T-60)**-0.0755)
    if T >= 66:
        b = 255
    elif T <= 19:
        b = 0
    else:
        b = 138.52*np.log(T-10) - 305.04
    return tuple(int(np.clip(x,0,255)) for x in (r,g,b))

colors = ['rgb{}'.format(blackbody_rgb(T)) for T in temps]

fig = go.Figure()
for i,(cl,T,ex,col) in enumerate(zip(classes,temps,examples,colors)):
    fig.add_shape(type='rect', x0=i, x1=i+1, y0=0, y1=1,
                  fillcolor=col, line=dict(color='white', width=2))
    fig.add_annotation(x=i+0.5, y=0.5, text=f'<b>{cl}</b>', showarrow=False,
                       font=dict(size=30, color='white' if T<7000 else 'black'))
    fig.add_annotation(x=i+0.5, y=1.12, text=f'{T:,} K', showarrow=False, font=dict(size=12))
    fig.add_annotation(x=i+0.5, y=-0.13, text=ex, showarrow=False, font=dict(size=11, color='#555'))
fig.update_xaxes(visible=False, range=[-0.2, 7.2])
fig.update_yaxes(visible=False, range=[-0.3, 1.3])
fig.update_layout(title='The Harvard spectral sequence: hot & blue (left) → cool & red (right)',
                  template='plotly_white', height=320, width=900, margin=dict(t=60,b=20))
fig.show()


### Demo 2 — how colour turns into a number

Telescopes don't record “bluish” — they record a **colour index**: brightness in a blue
filter minus brightness in a red filter. Gaia calls this **BP − RP**. A small (or negative)
index means a hot blue star; a large index means a cool red star. This demo shows the
smooth relationship between colour index and temperature.


In [ ]:
import plotly.express as px

# A schematic colour-index -> temperature curve (illustrative)
bp_rp = np.linspace(-0.4, 3.2, 300)
# reuse the same shape we use later for the real data
c = np.clip(bp_rp, -0.5, 5.0)
logT = 3.999 - 0.654*c + 0.709*c**2 - 0.316*c**3 + 0.0598*c**4 - 0.00413*c**5
teff = 10**logT

fig = px.line(x=bp_rp, y=teff,
              labels={'x':'Colour index  BP − RP', 'y':'Surface temperature (K)'},
              title='Colour index → temperature (the conversion behind the HR diagram)')
fig.update_traces(line=dict(width=4, color='#3b4c9a'))
# mark where each Harvard class roughly sits
marks = {'A':0.0,'F':0.4,'G':0.8,'K':1.4,'M':2.3}
for cl,x in marks.items():
    yy = float(10**(3.999 -0.654*x +0.709*x**2 -0.316*x**3 +0.0598*x**4 -0.00413*x**5))
    fig.add_scatter(x=[x], y=[yy], mode='markers+text', text=[cl], textposition='top center',
                    marker=dict(size=10, color='#b9770e'), showlegend=False)
fig.update_layout(template='plotly_white', height=460, width=760)
fig.show()
print('Notice: bluer (smaller BP-RP) = hotter. This is the x-axis of every HR diagram.')


### Demo 3 — a mini HR diagram with made-up stars

Finally, here's the *shape* we're aiming for, built from a handful of invented stars so you
recognise it when the real Gaia version appears. Colour on the x-axis, brightness on the y
(flipped so bright is up). Watch the main sequence, a giant, and a white dwarf appear.


In [ ]:
# made-up demo stars: (name, colour index, absolute magnitude, group)
demo_stars = [
    ('hot blue MS', -0.3, -2.0, 'main sequence'),
    ('white MS',     0.0,  1.0, 'main sequence'),
    ('Sun-like MS',  0.8,  4.8, 'main sequence'),
    ('orange MS',    1.3,  7.5, 'main sequence'),
    ('red dwarf MS', 2.0, 12.0, 'main sequence'),
    ('red giant',    1.6, -0.5, 'giant'),
    ('supergiant',   0.1, -6.0, 'giant'),
    ('white dwarf',  0.1, 12.5, 'white dwarf'),
]
import pandas as pd
d = pd.DataFrame(demo_stars, columns=['name','bp_rp','abs_mag','group'])

fig = px.scatter(d, x='bp_rp', y='abs_mag', color='group', text='name',
                 labels={'bp_rp':'Colour index  BP − RP','abs_mag':'Absolute magnitude'},
                 title='Mini demo HR diagram (made-up stars) — this is the shape to expect')
fig.update_traces(marker=dict(size=14), textposition='middle right')
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=520, width=820)
fig.show()
print('Bright + blue at top-left, faint + red at bottom-right = the main sequence.')


---
✅ **That's the whole idea.** Colour → temperature (x-axis), and true brightness (y-axis).
Now let's stop inventing numbers and pull **real stars** from ESA's Gaia archive. 👇


## 1. Get the data — from ESA's Gaia archive

Each Gaia data file is a compressed table of stars. The cell below downloads one chunk
(a few tens of thousands of stars) directly from ESA's servers — the same source
professional astronomers use.

It tries the live ESA download first. If the network is blocked (some networks restrict
this), it falls back to a Gaia-style sample so the rest of the notebook still runs.

In [ ]:
GAIA_URL = ('http://cdn.gea.esac.esa.int/Gaia/gedr3/gaia_source/'
            'GaiaSource_000000-003111.csv.gz')

def load_gaia(url=GAIA_URL):
    """Try the real ESA file; fall back to a realistic synthetic Gaia sample."""
    try:
        print('Downloading real Gaia data from ESA ... (this can take ~20–60 s)')
        df = pd.read_csv(url, comment='#')
        print(f'✅ Downloaded {len(df):,} real Gaia stars.')
        df.attrs['source'] = 'ESA Gaia EDR3 (live download)'
        return df
    except Exception as e:
        print('⚠️  Live download unavailable (', type(e).__name__, ').')
        print('   Generating a realistic Gaia-style sample so we can keep going...')
        return _synthetic_gaia()

def _synthetic_gaia(n=60000, seed=42):
    """A physically-motivated fake catalogue: main sequence + giants + white dwarfs.
    Column names match the real Gaia archive so all later code is identical."""
    rng = np.random.default_rng(seed)
    # population mix
    n_ms, n_giant, n_wd = int(0.82*n), int(0.13*n), n-int(0.82*n)-int(0.13*n)
    # --- main sequence: colour bp_rp vs absolute mag follow a slanted band ---
    bp_ms = rng.uniform(-0.4, 3.2, n_ms)
    Mabs_ms = 4.3*bp_ms + 1.0 + rng.normal(0, 0.45, n_ms)
    # --- red giant branch: red colour, bright (low Mabs) ---
    bp_g = rng.normal(1.25, 0.25, n_giant)
    Mabs_g = rng.normal(0.7, 0.9, n_giant) - 0.5*(bp_g-1.0)
    # --- white dwarfs: blue-ish but very faint (high Mabs) ---
    bp_wd = rng.uniform(-0.2, 0.9, n_wd)
    Mabs_wd = 11.5 + 2.5*bp_wd + rng.normal(0, 0.4, n_wd)
    bp_rp = np.concatenate([bp_ms, bp_g, bp_wd])
    Mabs  = np.concatenate([Mabs_ms, Mabs_g, Mabs_wd])
    # give every star a random distance (10–1500 pc) and back out parallax+apparent mag
    dist_pc = rng.uniform(10, 1500, len(bp_rp))
    parallax = 1000.0/dist_pc                          # mas
    g_app = Mabs - 5 - 5*np.log10(parallax/1000.0)
    parallax_error = np.abs(rng.normal(0.03, 0.02, len(bp_rp)))*parallax + 0.01
    ra  = rng.uniform(0, 360, len(bp_rp))
    dec = rng.uniform(-30, 30, len(bp_rp))
    df = pd.DataFrame({
        'source_id': np.arange(len(bp_rp)),
        'ra': ra, 'dec': dec,
        'parallax': parallax, 'parallax_error': parallax_error,
        'phot_g_mean_mag': g_app,
        'bp_rp': bp_rp,
        'phot_bp_mean_mag': g_app + bp_rp/2,
        'phot_rp_mean_mag': g_app - bp_rp/2,
    })
    df.attrs['source'] = 'Synthetic Gaia-style sample (offline fallback)'
    print(f'✅ Built {len(df):,} synthetic stars.')
    return df

df = load_gaia()
print('\nData source:', df.attrs.get('source'))
df.head()


### What are we even looking at?

Each row is one star. The columns that matter for us:

| Column | Meaning |
|---|---|
| `phot_g_mean_mag` | how **bright** the star looks from Earth (apparent magnitude) |
| `bp_rp` | the star's **colour** (blue brightness minus red brightness). Small = blue/hot, large = red/cool |
| `parallax` | the tiny wobble that tells us the **distance** (in milli-arcseconds) |
| `parallax_error` | the uncertainty on that distance |

The trick: *apparent* brightness depends on distance. A faint-looking star might just be far
away. To compare stars fairly we convert to **absolute magnitude** — how bright each star
*truly* is.


In [ ]:
# Absolute magnitude from apparent magnitude + parallax (the distance modulus).
# parallax is in milli-arcsec, so distance(pc) = 1000/parallax.
df['abs_mag'] = df['phot_g_mean_mag'] + 5 + 5*np.log10(df['parallax']/1000.0)
df['distance_pc'] = 1000.0/df['parallax']

print('Added two columns: abs_mag (true brightness) and distance_pc (distance in parsecs).')
df[['bp_rp','phot_g_mean_mag','parallax','abs_mag','distance_pc']].describe().round(2)


## 2. The raw HR diagram

Plot **colour** (x) against **absolute magnitude** (y). One convention: brighter stars have
*smaller* magnitude numbers, so we **flip the y-axis** — bright stars on top, faint on the
bottom, exactly how the sky is arranged.

👉 This plot is interactive: **drag to zoom**, double-click to reset, hover for star details.


In [ ]:
plot_df = df.dropna(subset=['bp_rp','abs_mag'])
# downsample for snappy interactivity if the catalogue is large
sample = plot_df.sample(min(25000, len(plot_df)), random_state=0)

fig = px.scatter(
    sample, x='bp_rp', y='abs_mag',
    title='Raw Hertzsprung–Russell Diagram (all stars, no quality cuts)',
    labels={'bp_rp':'Colour  BP − RP   (blue ← → red)',
            'abs_mag':'Absolute magnitude (bright ↑)'},
    opacity=0.4,
)
fig.update_traces(marker=dict(size=2.5, color='#3b82f6'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=650, width=850)
fig.show()


Notice the smear? Lots of stars scatter all over the place. That's not real astrophysics —
it's **noise** from stars whose distances we don't actually know well. Time to clean up.


## 3. Clean it up — keep only well-measured distances

A distance is only trustworthy if the parallax error is a small fraction of the parallax.
The standard cut is **`parallax_error / parallax < 5%`** (and positive parallax). Watch the
famous shape of the **main sequence** snap into focus.


In [ ]:
good = (df['parallax'] > 0) & (df['parallax_error']/df['parallax'] < 0.05)
clean = df[good].dropna(subset=['bp_rp','abs_mag'])
print(f'Kept {len(clean):,} of {len(df):,} stars ({100*len(clean)/len(df):.1f}%) after the 5% cut.')

s2 = clean.sample(min(25000, len(clean)), random_state=0)
fig = px.scatter(
    s2, x='bp_rp', y='abs_mag',
    title='Cleaned HR Diagram — parallax error < 5%',
    labels={'bp_rp':'Colour  BP − RP', 'abs_mag':'Absolute magnitude'},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=2.5, color='#6366f1'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(template='plotly_white', height=650, width=850)
fig.show()


## 4. The real HR diagram — coloured by temperature

Now the showpiece. We:
- convert Gaia colour into an approximate **surface temperature** (in Kelvin),
- colour every star by that temperature using a true blackbody-like palette (blue=hot,
  red=cool),
- and annotate the three great families of stars.

This is the diagram that won Hertzsprung and Russell their place in history — built by *you*
from raw spacecraft data.


In [ ]:
# A simple, well-known empirical relation mapping Gaia BP-RP colour -> effective temperature.
# (Good enough to teach with; professionals use fuller calibrations.)
def bp_rp_to_teff(bp_rp):
    c = np.clip(bp_rp, -0.5, 5.0)
    # polynomial fit in log-space, returns Kelvin
    logT = 3.999 - 0.654*c + 0.709*c**2 - 0.316*c**3 + 0.0598*c**4 - 0.00413*c**5
    return 10**logT

clean = clean.copy()
clean['teff'] = bp_rp_to_teff(clean['bp_rp'])
s3 = clean.sample(min(30000, len(clean)), random_state=1)

fig = px.scatter(
    s3, x='bp_rp', y='abs_mag', color='teff',
    color_continuous_scale='RdYlBu',  # red(cool) -> blue(hot) when we reverse
    range_color=(3000, 10000),
    labels={'bp_rp':'Colour  BP − RP', 'abs_mag':'Absolute magnitude',
            'teff':'T_eff (K)'},
    title='The Hertzsprung–Russell Diagram, coloured by stellar surface temperature',
    hover_data={'distance_pc':':.0f','teff':':.0f','bp_rp':':.2f','abs_mag':':.2f'},
)
fig.update_traces(marker=dict(size=3))
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(autorange='reversed')  # hot/blue on the LEFT, classic orientation

# ---- annotate the great stellar families ----
ann = dict(showarrow=False, font=dict(size=13, color='#111'),
           bgcolor='rgba(255,255,255,0.75)', bordercolor='#888', borderwidth=1)
fig.add_annotation(x=0.6, y=8.5, text='⭐ MAIN SEQUENCE<br>(hydrogen-burning, like the Sun)', **ann)
fig.add_annotation(x=1.7, y=0.5, text='🔴 RED GIANTS<br>(dying, bloated stars)', **ann)
fig.add_annotation(x=0.3, y=12.5, text='⚪ WHITE DWARFS<br>(stellar corpses)', **ann)
fig.add_annotation(x=0.0, y=4.8, text='☉ the Sun sits about here', font=dict(size=12,color='#b45309'),
                   showarrow=True, arrowhead=2, ax=40, ay=-30)

fig.update_layout(template='plotly_white', height=720, width=900,
                  coloraxis_colorbar=dict(title='T (K)'))
fig.show()


> **Note:** the main sequence isn't a stage of *age* — it's where a star spends about 90%
> of its life fusing hydrogen. A star's position *along* it is set mostly by its **mass**.
> The giants and white dwarfs are the same kinds of stars at later points in their lives.

## 5. The playground — move the sliders

Drag the sliders and the HR diagram redraws instantly. A few things to try:

- tighten the **parallax-quality** cut and watch the noise disappear
- limit the **distance** to keep only nearby stars
- change the **point size and opacity** for a denser or sparser look

In [ ]:
# --- Robust interactive playground (works in Jupyter AND Google Colab) ---
import plotly.graph_objects as go
from ipywidgets import interactive_output, FloatSlider, IntSlider, VBox, Output
from IPython.display import display

base = df.dropna(subset=['bp_rp','abs_mag']).copy()
base = base[base['parallax'] > 0]
base['teff'] = bp_rp_to_teff(base['bp_rp'])

# In Google Colab, enable the custom widget manager so plots render.
try:
    import google.colab  # noqa
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

out = Output()

def hr_playground(max_par_err_pct=5.0, max_distance_pc=1500, point_size=3, opacity=0.6):
    m = (base['parallax_error']/base['parallax'] < max_par_err_pct/100.0) & \
        (base['distance_pc'] <= max_distance_pc)
    d = base[m]
    if len(d) > 30000:
        d = d.sample(30000, random_state=0)
    fig = go.Figure(go.Scattergl(
        x=d['bp_rp'], y=d['abs_mag'], mode='markers',
        marker=dict(size=point_size, opacity=opacity, color=d['teff'],
                    colorscale='RdYlBu', cmin=3000, cmax=10000,
                    colorbar=dict(title='T (K)')),
        hovertemplate='colour=%{x:.2f}<br>abs mag=%{y:.2f}<extra></extra>',
    ))
    fig.update_yaxes(autorange='reversed', title='Absolute magnitude')
    fig.update_xaxes(autorange='reversed', title='Colour  BP − RP')
    fig.update_layout(template='plotly_white', height=620, width=850,
                      title=f'{len(d):,} stars  |  err<{max_par_err_pct:.1f}%  |  d<{max_distance_pc} pc')
    with out:
        out.clear_output(wait=True)
        fig.show()

s_err  = FloatSlider(value=5, min=1, max=50, step=1, description='max err %')
s_dist = IntSlider(value=1500, min=50, max=2000, step=50, description='max dist pc')
s_size = IntSlider(value=3, min=1, max=8, step=1, description='size')
s_op   = FloatSlider(value=0.6, min=0.1, max=1.0, step=0.1, description='opacity')

ui = VBox([s_err, s_dist, s_size, s_op])
interactive_output(hr_playground, dict(
    max_par_err_pct=s_err, max_distance_pc=s_dist,
    point_size=s_size, opacity=s_op))

hr_playground()        # draw once up front so a plot is visible immediately
display(ui, out)       # sliders, then the live plot below them

## 6. Where do the famous stars sit?

After the full crowd, here are a few well-known stars — the ones you can pick out in the
night sky. Where each one lands on the diagram tells you immediately what kind of star it is.

In [ ]:
# A few well-known stars with their approximate colour (BP-RP) and absolute
# magnitude, plus a one-line description of each.
famous_stars = [
    # name,            colour, abs_mag, short description
    ('☉ The Sun',        0.82,  4.83, 'Our home star — an ordinary main-sequence star'),
    ('Sirius A',         0.00,  1.45, 'Brightest star in our night sky; hot, white, nearby'),
    ('Betelgeuse',       1.85, -5.85, 'Red supergiant in Orion — may explode as a supernova'),
    ('Rigel',           -0.03, -7.0,  'Blue supergiant — thousands of times brighter than the Sun'),
    ('Vega',             0.00,  0.58, 'Bright blue-white star; was the old North Star'),
    ('Proxima Centauri', 2.95, 15.6,  'Closest star to the Sun — a tiny, faint red dwarf'),
    ('Sirius B',        -0.03, 11.3,  'A white dwarf — a dead stellar core the size of Earth'),
]
fam = pd.DataFrame(famous_stars, columns=['name','bp_rp','abs_mag','description'])

# Plot the real Gaia cloud faintly in the background, then drop the famous stars on top.
bg = clean.sample(min(20000, len(clean)), random_state=2)

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=bg['bp_rp'], y=bg['abs_mag'], mode='markers',
    marker=dict(size=2.5, color='lightgray', opacity=0.45),
    name='Gaia stars', hoverinfo='skip'))
fig.add_trace(go.Scatter(
    x=fam['bp_rp'], y=fam['abs_mag'], mode='markers+text',
    text=fam['name'], textposition='top center',
    textfont=dict(size=12, color='#111'),
    marker=dict(size=15, color=fam['bp_rp'], colorscale='RdYlBu_r',
                cmin=-0.2, cmax=3, line=dict(width=1.5, color='black')),
    customdata=fam['description'],
    hovertemplate='<b>%{text}</b><br>%{customdata}<extra></extra>',
    name='Famous stars'))
fig.update_yaxes(autorange='reversed', title='Absolute magnitude (bright ↑)')
fig.update_xaxes(autorange='reversed', title='Colour  BP − RP  (blue/hot ←  → red/cool)')
fig.update_layout(template='plotly_white', height=680, width=900,
                  title='Famous stars on the HR diagram (hover for details)',
                  legend=dict(x=0.02, y=0.02))
fig.show()

# Print the descriptions too, so they're visible without hovering.
print('Famous stars in this plot:\n')
for _, r in fam.iterrows():
    print(f"  {r['name']:<18}  colour={r['bp_rp']:+.2f}  abs_mag={r['abs_mag']:+.2f}")
    print(f"      → {r['description']}\n")


**Read the plot like an astronomer:**
- **Sirius A, Vega** sit on the upper main sequence — hot, bright, blue-white.
- **The Sun** sits modestly in the middle of the main sequence.
- **Proxima Centauri** is bottom-right — cool, red, and very faint.
- **Betelgeuse & Rigel** float top-right/top-left of everything — supergiants, off the main sequence.
- **Sirius B** hides at bottom-left — hot but tiny and faint: a white dwarf.

Same two numbers — colour and brightness — and you can already classify any star. 🌟


## 7. Animation — the life and death of a Sun-like star

The cell below renders a short MP4 of how a star like the Sun travels across the HR diagram
over billions of years: from a steady main-sequence star, to a swollen red giant, to a
puffed-off planetary nebula, and finally to a slowly cooling white dwarf.

Run the cell, wait a few seconds while it renders, and the video plays inline.

In [ ]:
# Builds an MP4 of a Sun-like (1 solar mass) star's journey across the HR diagram,
# then plays it inline. Physically ordered: main sequence -> giant -> planetary
# nebula -> cooling white dwarf, with a temperature-coloured fading trail.
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from IPython.display import HTML, display

# --- evolutionary phases for a 1 solar-mass star (colour BP-RP, absolute mag) ---
# Each phase is a short smooth segment, so the overall path follows the real
# shape astronomers draw instead of zig-zagging. The white dwarf correctly
# cools DOWN and to the RIGHT (redder) as it fades.
phases = [
    ("Main sequence — fusing hydrogen (today)",
        [(0.82, 4.83), (0.84, 4.70), (0.88, 4.45)]),
    ("Subgiant — hydrogen burning in a shell",
        [(0.88, 4.45), (0.98, 3.70), (1.10, 3.00)]),
    ("Red giant branch — swelling and cooling",
        [(1.10, 3.00), (1.30, 1.60), (1.48, 0.20), (1.58, -0.60)]),
    ("Helium flash — core helium ignites",
        [(1.58, -0.60), (1.30, 0.55), (1.05, 0.75)]),
    ("Horizontal branch — steady helium burning",
        [(1.05, 0.75), (1.15, 0.65), (1.28, 0.55)]),
    ("Asymptotic giant — huge and luminous",
        [(1.28, 0.55), (1.45, -0.60), (1.60, -1.60)]),
    ("Planetary nebula — outer layers ejected",
        [(1.60, -1.60), (1.00, -1.00), (0.30, -0.20), (-0.20, 1.50)]),
    ("White dwarf — cooling and fading forever",
        [(-0.20, 1.50), (-0.25, 5.00), (-0.10, 8.50), (0.20, 11.50), (0.50, 13.20)]),
]

# Build one dense, smooth polyline through all phases (numpy only, no scipy)
xs, ys, seg, labels = [], [], [], []
per_phase = 28
for pi, (lab, pts) in enumerate(phases):
    labels.append(lab)
    pts = np.array(pts, float)
    t  = np.linspace(0, 1, len(pts))
    tf = np.linspace(0, 1, per_phase)
    xs.append(np.interp(tf, t, pts[:, 0]))
    ys.append(np.interp(tf, t, pts[:, 1]))
    seg.append(np.full(per_phase, pi))
fx = np.concatenate(xs); fy = np.concatenate(ys); seg = np.concatenate(seg)
N = len(fx)

# Star's colour reflects its temperature (blue when hot, red when cool)
cnorm = Normalize(-0.3, 2.0)
star_cmap = matplotlib.colormaps['RdYlBu_r']

# Background: the real Gaia HR diagram, faint
bg = clean.sample(min(15000, len(clean)), random_state=7)

fig, ax = plt.subplots(figsize=(9, 6), dpi=120)
ax.scatter(bg['bp_rp'], bg['abs_mag'], s=2, c='lightgray', alpha=0.4, zorder=1)
ax.set_xlim(-0.6, 2.1)
ax.set_ylim(14.5, -3)          # inverted: bright at top
ax.set_xlabel('Colour  BP − RP   (blue/hot ←   → red/cool)')
ax.set_ylabel('Absolute magnitude (bright ↑)')
ax.set_title('The life of a Sun-like star across the HR diagram')
ax.grid(alpha=0.15)

trail = LineCollection([], linewidths=2.6, zorder=2)
ax.add_collection(trail)
glow,     = ax.plot([], [], 'o', ms=34, color='gold', alpha=0.22, zorder=3)
star_dot, = ax.plot([], [], 'o', ms=19, mec='black', mew=1.2, zorder=4)
stage_txt = ax.text(0.03, 0.06, '', transform=ax.transAxes, fontsize=13,
                    weight='bold',
                    bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#444'))

def update(i):
    if i > 0:
        pts = np.column_stack([fx[:i+1], fy[:i+1]])
        segs = np.stack([pts[:-1], pts[1:]], axis=1)
        trail.set_segments(segs)
        cols = star_cmap(cnorm(fx[:i]))      # colour by temperature along the trail
        cols[:, 3] = np.linspace(0.20, 1.0, i)   # older parts fade out
        trail.set_color(cols)
    c = star_cmap(cnorm(fx[i]))
    star_dot.set_data([fx[i]], [fy[i]]); star_dot.set_color(c)
    glow.set_data([fx[i]], [fy[i]])
    stage_txt.set_text(labels[seg[i]])
    return trail, glow, star_dot, stage_txt

anim = FuncAnimation(fig, update, frames=N, interval=60, blit=False)

# Save to MP4 (needs ffmpeg — preinstalled on Colab; 'apt-get install ffmpeg' if missing)
out_path = 'sun_like_star_evolution.mp4'
try:
    anim.save(out_path, writer=FFMpegWriter(fps=18, bitrate=2400))
    plt.close(fig)
    print('✅ Video saved as', out_path, '— playing below:')
    import base64
    with open(out_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'''
        <video width="760" controls autoplay loop>
          <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        </video>'''))
except Exception as e:
    plt.close(fig)
    print('MP4 writer unavailable (', type(e).__name__, ').')
    print('On Colab run:  !apt-get -qq install ffmpeg   then re-run this cell.')
    print('Falling back to an inline JS animation instead...')
    display(HTML(anim.to_jshtml()))

> A star doesn't drift smoothly across the whole diagram. It spends almost its entire life
> on the main sequence, then moves quickly (in cosmic terms) up into the giant region, sheds
> its outer layers, and ends as a cooling white dwarf in the lower left. The HR diagram is,
> in effect, a map of stellar fate.

## 8. Bonus — read a cluster's age with isochrones

An **isochrone** is the HR-diagram track of a group of stars that were all born at the same
time. As a cluster ages, its most massive (bluest) stars die first, so the top of the main
sequence peels away — the **turn-off** point slides down and to the right. The position of
that turn-off tells you the cluster's age.

The cell below overlays a few schematic isochrones on the data. They are illustrative tracks
for intuition, not a fitted stellar model.

In [ ]:
def schematic_isochrone(age_gyr, n=200):
    """Toy isochrone: a main sequence that turns off earlier (bluer) for younger ages."""
    # turn-off colour gets redder (larger bp_rp) with age
    turnoff = 0.2 + 0.35*np.log10(age_gyr*1000)  # crude but monotonic
    bp = np.linspace(-0.3, 3.0, n)
    Mabs = 4.3*bp + 1.0  # the main-sequence line from earlier
    # above the turn-off, bend up toward the giant branch
    giant = bp > turnoff
    Mabs[giant] = Mabs[giant] - (bp[giant]-turnoff)*6.0
    return bp, Mabs

s4 = clean.sample(min(15000, len(clean)), random_state=5)
fig = go.Figure()
fig.add_trace(go.Scattergl(x=s4['bp_rp'], y=s4['abs_mag'], mode='markers',
              marker=dict(size=2.5, color='lightgray', opacity=0.5), name='Gaia stars'))

colors = px.colors.sequential.Plasma
for i, age in enumerate([0.1, 0.5, 1, 5, 10]):
    bp, M = schematic_isochrone(age)
    fig.add_trace(go.Scatter(x=bp, y=M, mode='lines',
                  line=dict(width=3, color=colors[int(i/4*(len(colors)-1))]),
                  name=f'{age} Gyr'))

fig.update_yaxes(autorange='reversed', title='Absolute magnitude')
fig.update_xaxes(title='Colour  BP − RP')
fig.update_layout(template='plotly_white', height=680, width=880,
                  title='Schematic isochrones over the Gaia HR diagram — older = redder turn-off',
                  legend_title='Age')
fig.show()


## 9. Challenge — extend the notebook

The pipeline works end to end. Here are a few ways to take it further:

1. **Different sky patch.** Swap `GAIA_URL` for another file from the
   [ESA Gaia archive](https://gea.esac.esa.int/archive/) (see the file list under
   `gaia_source/`). Does the HR diagram change shape?
2. **Density instead of dots.** Replace the scatter in section 4 with
   `px.density_heatmap(...)` to see where stars pile up.
3. **Find the Sun's twins.** Filter `clean` to stars within 0.1 mag and 0.05 colour of the
   Sun (abs_mag ≈ 4.83, bp_rp ≈ 0.82) and count them.
4. **Colour–magnitude of a real cluster.** Run a cone search around the Pleiades and watch a
   single tight isochrone appear instead of a smear.
5. **Better temperature scale.** Improve `bp_rp_to_teff` with a published Gaia calibration
   and recolour the diagram.

A starter cell follows.

In [ ]:
# Example: find Sun-like stars in the cleaned catalogue.
sun_like = clean[(clean['abs_mag'].between(4.6, 5.0)) &
                 (clean['bp_rp'].between(0.75, 0.90))]
print(f'Found {len(sun_like):,} Sun-like stars in this sample.')
sun_like[['bp_rp','abs_mag','distance_pc','teff']].head(10).round(2)


---
# Project — rebuild this with a different telescope

The project is to repeat everything above using data from a **different telescope**. The
part that trips most people up is downloading the data and finding the right columns, so this
section walks through it carefully. The goal is a table with the **same four columns** the
Gaia HR diagram used.

### The four columns every HR diagram needs

| Role | What it is | Gaia name |
|------|-----------|-----------|
| brightness | apparent magnitude | `phot_g_mean_mag` |
| colour | colour index (x-axis) | `bp_rp` |
| distance | parallax in mas | `parallax` |
| quality | parallax error | `parallax_error` |

Whatever telescope you pick, the task is to find these four columns and rename them to match,
so the rest of the code works unchanged.

### Easiest path — ESA Hipparcos via `astroquery` (no website needed)

The cell below downloads Hipparcos, renames its columns to match Gaia's, and builds the same
`bp_rp` and `abs_mag` columns used earlier. From there, the rest of the project is yours.

In [ ]:
!pip install astroquery

In [ ]:
# Download Hipparcos and set up the same columns as the Gaia pipeline.
# If astroquery is missing:  pip install astroquery
import numpy as np, pandas as pd

try:
    from astroquery.vizier import Vizier
    Vizier.ROW_LIMIT = -1                       # no row cap: get every star
    print('Downloading Hipparcos catalogue (I/239/hip_main) ...')
    cat = Vizier.get_catalogs('I/239/hip_main')
    hip = cat[0].to_pandas()
    print(f'✅ Got {len(hip):,} Hipparcos stars.')

    # --- Hipparcos column names -> the names the HR-diagram code expects ---
    # Vmag = apparent magnitude, B-V = colour, Plx = parallax (mas), e_Plx = its error
    hip = hip.rename(columns={
        'Vmag':  'app_mag',
        'B-V':   'bp_rp',          # B-V plays the same role as Gaia's BP-RP
        'Plx':   'parallax',
        'e_Plx': 'parallax_error',
    })

    # Keep only the columns we need and drop blanks
    hip = hip[['app_mag','bp_rp','parallax','parallax_error']].dropna()
    hip = hip[hip['parallax'] > 0]

    # Same physics as before: distance + absolute magnitude
    hip['distance_pc'] = 1000.0/hip['parallax']
    hip['abs_mag'] = hip['app_mag'] + 5 + 5*np.log10(hip['parallax']/1000.0)

    print('\nThe Hipparcos table now has the same columns as the Gaia one:')
    print('   bp_rp, abs_mag, parallax, parallax_error, distance_pc')
    display(hip.head())

except Exception as e:
    print('Could not auto-download (', type(e).__name__, ':', e, ')')
    print('No problem — use the manual route in the next cell instead.')


### Backup path: download a CSV by hand (if `astroquery` won't install)
1. Go to **VizieR**: https://vizier.cds.unistra.fr
2. Search **`I/239`** (classic Hipparcos) or **`I/311`** (the improved re-reduction).
3. Open `hip_main`, set the output limit to *unlimited*, choose **CSV**, and download.
4. Upload the file here (in Colab: the folder icon on the left → upload), then run the cell
   below, editing the filename and column names if needed.


In [ ]:
# Manual CSV route — edit the filename to match what you downloaded.
import numpy as np, pandas as pd

# hip = pd.read_csv('hip_main.csv', comment='#')
#
# # Rename your file's columns to the standard four (check the file's header!):
# hip = hip.rename(columns={
#     'Vmag':  'app_mag',
#     'B-V':   'bp_rp',
#     'Plx':   'parallax',
#     'e_Plx': 'parallax_error',
# })
# hip = hip[['app_mag','bp_rp','parallax','parallax_error']].dropna()
# hip = hip[hip['parallax'] > 0]
# hip['distance_pc'] = 1000/hip['parallax']
# hip['abs_mag'] = hip['app_mag'] + 5 + 5*np.log10(hip['parallax']/1000)
# hip.head()
print('Uncomment the lines above once your CSV is uploaded.')


### Now it's your turn

You now have a `hip` table with `bp_rp`, `abs_mag`, `parallax`, and `parallax_error` —
exactly what the Gaia code used. Reuse the earlier cells almost verbatim to:

1. apply a quality cut (start with `parallax_error / parallax < 0.10`; Hipparcos is less
   precise than Gaia, so 10% is reasonable)
2. plot the raw HR diagram (`x='bp_rp'`, `y='abs_mag'`, with the y-axis flipped)
3. make one interactive or temperature-coloured version
4. write a few sentences comparing your Hipparcos diagram to the Gaia one

Same science, your own data.